<a href="https://colab.research.google.com/github/shraddhasaste9676-ai/docketIQ/blob/main/VORTEXAdatabase%26backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ipython-sql


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 59.5 MB/s eta 0:00:00


In [ ]:
-- ============================================================
-- CaseCompass: Legal Case Timeline & Action Organizer
-- PostgreSQL 14+
-- ============================================================

CREATE EXTENSION IF NOT EXISTS pg_trgm;
CREATE EXTENSION IF NOT EXISTS btree_gin;

-- ------------------------------------------------------------
-- 1. Cases
-- ------------------------------------------------------------
CREATE TABLE cases (
    case_id          UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    case_reference   TEXT NOT NULL,
    title            TEXT NOT NULL,
    jurisdiction     TEXT,
    court_or_tribunal TEXT,
    created_at       TIMESTAMPTZ NOT NULL DEFAULT now(),
    updated_at       TIMESTAMPTZ NOT NULL DEFAULT now(),

    CONSTRAINT uq_cases_reference UNIQUE (case_reference),
    CONSTRAINT ck_cases_title_nonempty CHECK (length(btrim(title)) > 0)
);

COMMENT ON TABLE cases IS
'A supplied case workspace. CaseCompass does not connect to live court systems.';

-- ------------------------------------------------------------
-- 2. Documents supplied by the user
-- ------------------------------------------------------------
CREATE TABLE documents (
    document_id      UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    case_id          UUID NOT NULL REFERENCES cases(case_id) ON DELETE CASCADE,
    document_type    TEXT NOT NULL,
    title            TEXT NOT NULL,
    file_name        TEXT,
    storage_uri      TEXT,
    sha256_hash      TEXT,
    document_date_raw TEXT,
    document_date    DATE,
    date_confidence  TEXT NOT NULL DEFAULT 'unparsed'
                     CHECK (date_confidence IN (
                         'high', 'medium', 'low', 'unparsed', 'conflict'
                     )),
    status           TEXT NOT NULL DEFAULT 'uploaded'
                     CHECK (status IN (
                         'uploaded', 'processing', 'processed',
                         'needs_review', 'error'
                     )),
    uploaded_at      TIMESTAMPTZ NOT NULL DEFAULT now(),

    CONSTRAINT uq_documents_case_hash UNIQUE (case_id, sha256_hash),
    CONSTRAINT ck_documents_type_nonempty CHECK (length(btrim(document_type)) > 0)
);

CREATE INDEX idx_documents_case ON documents(case_id);
CREATE INDEX idx_documents_date ON documents(document_date);
CREATE INDEX idx_documents_status ON documents(status);

COMMENT ON COLUMN documents.document_date_raw IS
'Original date text exactly as it appears in the document. Never overwritten.';

COMMENT ON COLUMN documents.document_date IS
'Normalized date stored only when parsing confidence is sufficient.';

-- ------------------------------------------------------------
-- 3. Pages / segments, preserving document position
-- ------------------------------------------------------------
CREATE TABLE document_pages (
    page_id          UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    document_id      UUID NOT NULL REFERENCES documents(document_id) ON DELETE CASCADE,
    page_number      INTEGER NOT NULL,
    page_text        TEXT NOT NULL,
    ocr_confidence   NUMERIC(4,3)
                     CHECK (ocr_confidence BETWEEN 0 AND 1),
    created_at       TIMESTAMPTZ NOT NULL DEFAULT now(),

    CONSTRAINT uq_documents_page_number UNIQUE (document_id, page_number),
    CONSTRAINT ck_page_number_positive CHECK (page_number > 0)
);

CREATE INDEX idx_pages_document ON document_pages(document_id);
CREATE INDEX idx_pages_text_trgm ON document_pages USING gin (page_text gin_trgm_ops);

-- ------------------------------------------------------------
-- 4. Exact source spans
-- ------------------------------------------------------------
CREATE TABLE source_spans (
    source_span_id   UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    document_id      UUID NOT NULL REFERENCES documents(document_id) ON DELETE CASCADE,
    page_id          UUID REFERENCES document_pages(page_id) ON DELETE SET NULL,
    start_offset     INTEGER NOT NULL,
    end_offset       INTEGER NOT NULL,
    source_text      TEXT NOT NULL,
    extraction_method TEXT NOT NULL DEFAULT 'manual'
                     CHECK (extraction_method IN (
                         'manual', 'ocr', 'ai', 'user_import'
                     )),
    created_at       TIMESTAMPTZ NOT NULL DEFAULT now(),

    CONSTRAINT ck_span_offsets CHECK (start_offset >= 0 AND end_offset > start_offset),
    CONSTRAINT ck_source_text_nonempty CHECK (length(btrim(source_text)) > 0)
);

CREATE INDEX idx_spans_document ON source_spans(document_id);
CREATE INDEX idx_spans_page ON source_spans(page_id);

COMMENT ON TABLE source_spans IS
'Immutable verbatim evidence. Every derived item must reference one or more spans.';

-- ------------------------------------------------------------
-- 5. Canonical case events
-- ------------------------------------------------------------
CREATE TABLE case_events (
    event_id         UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    case_id          UUID NOT NULL REFERENCES cases(case_id) ON DELETE CASCADE,
    event_type       TEXT NOT NULL,
    title            TEXT NOT NULL,
    description      TEXT,
    event_date       DATE,
    event_date_raw   TEXT,
    date_confidence  TEXT NOT NULL DEFAULT 'unparsed'
                     CHECK (date_confidence IN (
                         'high', 'medium', 'low', 'unparsed', 'conflict'
                     )),
    sequence_hint    INTEGER,
    is_order         BOOLEAN NOT NULL DEFAULT false,
    order_number     TEXT,
    review_status    TEXT NOT NULL DEFAULT 'pending'
                     CHECK (review_status IN (
                         'pending', 'ai_proposed', 'confirmed', 'rejected'
                     )),
    created_by       TEXT NOT NULL DEFAULT 'system',
    created_at       TIMESTAMPTZ NOT NULL DEFAULT now(),
    updated_at       TIMESTAMPTZ NOT NULL DEFAULT now(),

    CONSTRAINT ck_event_title_nonempty CHECK (length(btrim(title)) > 0)
);

CREATE INDEX idx_events_case_date ON case_events(case_id, event_date NULLS LAST);
CREATE INDEX idx_events_case_sequence ON case_events(case_id, sequence_hint NULLS LAST);
CREATE INDEX idx_events_type ON case_events(case_id, event_type);
CREATE INDEX idx_events_review ON case_events(review_status);

COMMENT ON TABLE case_events IS
'Reviewed chronological events. Dates are stored only when supported by evidence.';

-- ------------------------------------------------------------
-- 6. Event assertions: what each document says happened
-- ------------------------------------------------------------
CREATE TABLE event_assertions (
    assertion_id     UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    case_id          UUID NOT NULL REFERENCES cases(case_id) ON DELETE CASCADE,
    event_id         UUID REFERENCES case_events(event_id) ON DELETE SET NULL,
    document_id      UUID NOT NULL REFERENCES documents(document_id) ON DELETE CASCADE,
    source_span_id   UUID NOT NULL REFERENCES source_spans(source_span_id) ON DELETE RESTRICT,
    asserted_event   TEXT NOT NULL,
    asserted_date_raw TEXT,
    asserted_date    DATE,
    date_confidence  TEXT NOT NULL DEFAULT 'unparsed'
                     CHECK (date_confidence IN (
                         'high', 'medium', 'low', 'unparsed', 'conflict'
                     )),
    assertion_role   TEXT NOT NULL DEFAULT 'primary'
                     CHECK (assertion_role IN (
                         'primary', 'supporting', 'contradicting', 'reference_only'
                     )),
    ai_generated     BOOLEAN NOT NULL DEFAULT false,
    review_status    TEXT NOT NULL DEFAULT 'pending'
                     CHECK (review_status IN (
                         'pending', 'confirmed', 'rejected'
                     )),
    created_at       TIMESTAMPTZ NOT NULL DEFAULT now(),

    CONSTRAINT uq_event_assertion_span UNIQUE (source_span_id, asserted_event)
);

CREATE INDEX idx_assertions_case ON event_assertions(case_id);
CREATE INDEX idx_assertions_event ON event_assertions(event_id);
CREATE INDEX idx_assertions_document ON event_assertions(document_id);

-- ------------------------------------------------------------
-- 7. Date conflicts
-- ------------------------------------------------------------
CREATE TABLE event_conflicts (
    conflict_id      UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    case_id          UUID NOT NULL REFERENCES cases(case_id) ON DELETE CASCADE,
    event_id         UUID NOT NULL REFERENCES case_events(event_id) ON DELETE CASCADE,
    conflict_type    TEXT NOT NULL
                     CHECK (conflict_type IN (
                         'different_dates_same_event',
                         'ambiguous_date',
                         'missing_date',
                         'sequence_mismatch',
                         'order_reference_mismatch'
                     )),
    detected_at      TIMESTAMPTZ NOT NULL DEFAULT now(),
    resolution_status TEXT NOT NULL DEFAULT 'open'
                     CHECK (resolution_status IN (
                         'open', 'resolved', 'ignored'
                     )),
    resolution_note  TEXT,
    resolved_event_date DATE,
    resolved_by      TEXT,
    resolved_at      TIMESTAMPTZ
);

CREATE INDEX idx_conflicts_case ON event_conflicts(case_id);
CREATE INDEX idx_conflicts_event ON event_conflicts(event_id);
CREATE INDEX idx_conflicts_status ON event_conflicts(resolution_status);

CREATE TABLE event_conflict_assertions (
    conflict_id      UUID NOT NULL REFERENCES event_conflicts(conflict_id) ON DELETE CASCADE,
    assertion_id     UUID NOT NULL REFERENCES event_assertions(assertion_id) ON DELETE CASCADE,
    PRIMARY KEY (conflict_id, assertion_id)
);

-- ------------------------------------------------------------
-- 8. Explicit deadlines only
-- ------------------------------------------------------------
CREATE TABLE case_deadlines (
    deadline_id      UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    case_id          UUID NOT NULL REFERENCES cases(case_id) ON DELETE CASCADE,
    event_id         UUID REFERENCES case_events(event_id) ON DELETE SET NULL,
    title            TEXT NOT NULL,
    action_required  TEXT NOT NULL,
    due_date_raw     TEXT NOT NULL,
    due_date         DATE,
    date_confidence  TEXT NOT NULL DEFAULT 'unparsed'
                     CHECK (date_confidence IN (
                         'high', 'medium', 'low', 'unparsed', 'conflict'
                     )),
    deadline_basis   TEXT NOT NULL
                     CHECK (deadline_basis IN (
                         'explicit_in_record', 'user_supplied_rule'
                     )),
    user_rule_id     UUID,
    review_status    TEXT NOT NULL DEFAULT 'pending'
                     CHECK (review_status IN (
                         'pending', 'ai_proposed', 'confirmed', 'rejected'
                     )),
    created_at       TIMESTAMPTZ NOT NULL DEFAULT now(),
    updated_at       TIMESTAMPTZ NOT NULL DEFAULT now(),

    CONSTRAINT ck_deadline_basis CHECK (
        (deadline_basis = 'explicit_in_record')
        OR (deadline_basis = 'user_supplied_rule' AND user_rule_id IS NOT NULL)
    )
);

CREATE INDEX idx_deadlines_case_due ON case_deadlines(case_id, due_date NULLS LAST);
CREATE INDEX idx_deadlines_review ON case_deadlines(review_status);

COMMENT ON TABLE case_deadlines IS
'Only explicitly stated deadlines or user-supplied rule results. No system-calculated deadlines.';

CREATE TABLE deadline_assertions (
    deadline_assertion_id UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    deadline_id      UUID NOT NULL REFERENCES case_deadlines(deadline_id) ON DELETE CASCADE,
    document_id      UUID NOT NULL REFERENCES documents(document_id) ON DELETE CASCADE,
    source_span_id   UUID NOT NULL REFERENCES source_spans(source_span_id) ON DELETE RESTRICT,
    stated_due_date_raw TEXT NOT NULL,
    stated_due_date  DATE,
    date_confidence  TEXT NOT NULL DEFAULT 'unparsed'
                     CHECK (date_confidence IN (
                         'high', 'medium', 'low', 'unparsed', 'conflict'
                     )),
    ai_generated     BOOLEAN NOT NULL DEFAULT false,
    review_status    TEXT NOT NULL DEFAULT 'pending'
                     CHECK (review_status IN (
                         'pending', 'confirmed', 'rejected'
                     )),
    created_at       TIMESTAMPTZ NOT NULL DEFAULT now(),

    CONSTRAINT uq_deadline_span UNIQUE (source_span_id, stated_due_date_raw)
);

CREATE INDEX idx_deadline_assertions_case ON deadline_assertions(deadline_id);
CREATE INDEX idx_deadline_assertions_document ON deadline_assertions(document_id);

-- ------------------------------------------------------------
-- 9. User-supplied deadline rules
-- ------------------------------------------------------------
CREATE TABLE user_rules (
    user_rule_id     UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    case_id          UUID NOT NULL REFERENCES cases(case_id) ON DELETE CASCADE,
    rule_name        TEXT NOT NULL,
    rule_text        TEXT NOT NULL,
    input_event_id   UUID REFERENCES case_events(event_id) ON DELETE SET NULL,
    input_date       DATE,
    computed_due_date DATE,
    rule_version     INTEGER NOT NULL DEFAULT 1,
    confirmed_by_user BOOLEAN NOT NULL DEFAULT false,
    created_at       TIMESTAMPTZ NOT NULL DEFAULT now(),

    CONSTRAINT ck_user_rule_confirmed CHECK (
        computed_due_date IS NULL OR confirmed_by_user = true
    )
);

ALTER TABLE case_deadlines
    ADD CONSTRAINT fk_deadlines_user_rule
    FOREIGN KEY (user_rule_id)
    REFERENCES user_rules(user_rule_id) ON DELETE SET NULL;

-- ------------------------------------------------------------
-- 10. Missing / referenced orders and gaps
-- ------------------------------------------------------------
CREATE TABLE order_references (
    order_reference_id UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    case_id          UUID NOT NULL REFERENCES cases(case_id) ON DELETE CASCADE,
    source_span_id   UUID NOT NULL REFERENCES source_spans(source_span_id) ON DELETE RESTRICT,
    referenced_order_text TEXT NOT NULL,
    referenced_order_number TEXT,
    referenced_date_raw TEXT,
    referenced_date  DATE,
    matching_document_id UUID REFERENCES documents(document_id) ON DELETE SET NULL,
    match_status     TEXT NOT NULL DEFAULT 'unmatched'
                     CHECK (match_status IN (
                         'unmatched', 'probable_match', 'matched', 'missing'
                     )),
    ai_generated     BOOLEAN NOT NULL DEFAULT false,
    review_status    TEXT NOT NULL DEFAULT 'pending'
                     CHECK (review_status IN (
                         'pending', 'confirmed', 'rejected'
                     )),
    created_at       TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE INDEX idx_order_refs_case ON order_references(case_id);
CREATE INDEX idx_order_refs_status ON order_references(match_status);

CREATE TABLE timeline_gaps (
    gap_id           UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    case_id          UUID NOT NULL REFERENCES cases(case_id) ON DELETE CASCADE,
    gap_type         TEXT NOT NULL
                     CHECK (gap_type IN (
                         'missing_referenced_order',
                         'event_without_date',
                         'possible_missing_step',
                         'document_date_conflict',
                         'unmatched_deadline',
                         'unreviewed_ai_item'
                     )),
    related_event_id UUID REFERENCES case_events(event_id) ON DELETE SET NULL,
    related_document_id UUID REFERENCES documents(document_id) ON DELETE SET NULL,
    related_order_reference_id UUID REFERENCES order_references(order_reference_id) ON DELETE SET NULL,
    related_conflict_id UUID REFERENCES event_conflicts(conflict_id) ON DELETE SET NULL,
    gap_description  TEXT NOT NULL,
    severity         TEXT NOT NULL DEFAULT 'medium'
                     CHECK (severity IN ('low', 'medium', 'high')),
    status           TEXT NOT NULL DEFAULT 'open'
                     CHECK (status IN ('open', 'resolved', 'ignored')),
    created_at       TIMESTAMPTZ NOT NULL DEFAULT now(),
    resolved_at      TIMESTAMPTZ,
    resolution_note  TEXT
);

CREATE INDEX idx_gaps_case_status ON timeline_gaps(case_id, status);
CREATE INDEX idx_gaps_type ON timeline_gaps(case_id, gap_type);

-- ------------------------------------------------------------
-- 11. AI extraction queue
-- ------------------------------------------------------------
CREATE TABLE ai_extractions (
    ai_extraction_id UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    case_id          UUID NOT NULL REFERENCES cases(case_id) ON DELETE CASCADE,
    document_id      UUID NOT NULL REFERENCES documents(document_id) ON DELETE CASCADE,
    source_span_id   UUID NOT NULL REFERENCES source_spans(source_span_id) ON DELETE RESTRICT,
    extraction_type  TEXT NOT NULL
                     CHECK (extraction_type IN (
                         'event', 'deadline', 'order_reference',
                         'date', 'terminology', 'relationship'
                     )),
    proposed_payload JSONB NOT NULL,
    model_name       TEXT,
    model_version    TEXT,
    confidence       NUMERIC(4,3)
                     CHECK (confidence BETWEEN 0 AND 1),
    validation_status TEXT NOT NULL DEFAULT 'pending'
                     CHECK (validation_status IN (
                         'pending', 'valid', 'invalid_no_source',
                         'invalid_low_confidence', 'rejected', 'accepted'
                     )),
    validation_errors JSONB NOT NULL DEFAULT '[]'::jsonb,
    reviewed_by      TEXT,
    reviewed_at      TIMESTAMPTZ,
    created_at       TIMESTAMPTZ NOT NULL DEFAULT now(),

    CONSTRAINT ck_ai_payload_is_object CHECK (
        jsonb_typeof(proposed_payload) = 'object'
    )
);

CREATE INDEX idx_ai_case_status ON ai_extractions(case_id, validation_status);
CREATE INDEX idx_ai_document ON ai_extractions(document_id);
CREATE INDEX idx_ai_payload_gin ON ai_extractions USING gin (proposed_payload);

COMMENT ON TABLE ai_extractions IS
'AI output is never inserted directly into the reviewed timeline until accepted and source-linked.';

-- ------------------------------------------------------------
-- 12. Glossary / terminology explanations
-- ------------------------------------------------------------
CREATE TABLE terminology_notes (
    term_id          UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    case_id          UUID NOT NULL REFERENCES cases(case_id) ON DELETE CASCADE,
    term             TEXT NOT NULL,
    explanation      TEXT NOT NULL,
    source_span_id   UUID REFERENCES source_spans(source_span_id) ON DELETE SET NULL,
    ai_generated     BOOLEAN NOT NULL DEFAULT false,
    review_status    TEXT NOT NULL DEFAULT 'pending'
                     CHECK (review_status IN (
                         'pending', 'confirmed', 'rejected'
                     )),
    created_at       TIMESTAMPTZ NOT NULL DEFAULT now(),

    CONSTRAINT uq_case_term UNIQUE (case_id, term)
);

-- ------------------------------------------------------------
-- 13. Audit trail
-- ------------------------------------------------------------
CREATE TABLE audit_log (
    audit_id         BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    case_id          UUID REFERENCES cases(case_id) ON DELETE SET NULL,
    actor            TEXT NOT NULL,
    actor_type       TEXT NOT NULL
                     CHECK (actor_type IN ('user', 'system', 'ai')),
    action           TEXT NOT NULL,
    entity_type      TEXT NOT NULL,
    entity_id        UUID,
    before_state     JSONB,
    after_state      JSONB,
    occurred_at      TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE INDEX idx_audit_case_time ON audit_log(case_id, occurred_at DESC);

-- ------------------------------------------------------------
-- 14. Updated-at triggers
-- ------------------------------------------------------------
CREATE OR REPLACE FUNCTION set_updated_at()
RETURNS TRIGGER AS $$
BEGIN
    NEW.updated_at := now();
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;

CREATE TRIGGER trg_cases_updated
BEFORE UPDATE ON cases
FOR EACH ROW EXECUTE FUNCTION set_updated_at();

CREATE TRIGGER trg_events_updated
BEFORE UPDATE ON case_events
FOR EACH ROW EXECUTE FUNCTION set_updated_at();

CREATE TRIGGER trg_deadlines_updated
BEFORE UPDATE ON case_deadlines
FOR EACH ROW EXECUTE FUNCTION set_updated_at();

SyntaxError: invalid syntax (3909415206.py, line 1)

In [ ]:
%%bash
sudo apt-get -qq update
sudo apt-get install -y postgresql postgresql-contrib
sudo service postgresql start

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libcommon-sense-perl libjson-perl libjson-xs-perl libtypes-serialiser-perl
  logrotate postgresql-16 postgresql-client-16 postgresql-client-common
  postgresql-common ssl-cert sysstat
Suggested packages:
  bsd-mailx | mailx postgresql-doc postgresql-doc-16 isag
The following NEW packages will be installed:
  libcommon-sense-perl libjson-perl libjson-xs-perl libtypes-serialiser-perl
  logrotate postgresql postgresql-16 postgresql-client-16
  postgresql-client-common postgresql-common postgresql-contrib ssl-cert
  sysstat
0 upgraded, 13 newly installed, 0 to remove and 24 not upgraded.
Need to get 17.9 MB of archives.
After this operation, 52.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 libjson-perl all 4.10000-1 [81.9 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 postgresql-cl

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 13.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


In [ ]:
%%bash
sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'casecompass';"
sudo -u postgres psql -c "CREATE DATABASE casecompass_db;"

ALTER ROLE


ERROR:  database "casecompass_db" already exists


CalledProcessError: Command 'b'sudo -u postgres psql -c "ALTER USER postgres PASSWORD \'casecompass\';"\nsudo -u postgres psql -c "CREATE DATABASE casecompass_db;"\n'' returned non-zero exit status 1.

In [ ]:
%%bash
sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'casecompass';"
sudo -u postgres psql -c "CREATE DATABASE casecompass_db;"

ALTER ROLE


ERROR:  database "casecompass_db" already exists


CalledProcessError: Command 'b'sudo -u postgres psql -c "ALTER USER postgres PASSWORD \'casecompass\';"\nsudo -u postgres psql -c "CREATE DATABASE casecompass_db;"\n'' returned non-zero exit status 1.

In [ ]:
from datasets import load_dataset

ds = load_dataset("KanoonGPT/indian-legal-documents")

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

data/train-00000.parquet: reconstructing file:   0%|          |  0.00B / 56.8MB            

data/train-00000.parquet: downloading bytes:           |  0.00B            

data/train-00001.parquet: reconstructing file:   0%|          |  0.00B / 66.5MB            

data/train-00001.parquet: downloading bytes:           |  0.00B            

...

data/train-00006.parquet: reconstructing file:   0%|          |  0.00B /  100MB            

data/train-00006.parquet: downloading bytes:           |  0.00B            

data/train-00007.parquet: reconstructing file:   0%|          |  0.00B / 19.0MB            

data/train-00007.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/35845 [00:00<?, ? examples/s]

In [ ]:
sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'casecompass';"


SyntaxError: invalid syntax (2961439138.py, line 1)

In [ ]:
%%bash
sudo apt-get -qq update
sudo apt-get install -y postgresql postgresql-contrib
sudo service postgresql start
sudo -u postgres psql -c "SELECT version();"

Reading package lists...
Building dependency tree...
Reading state information...
postgresql is already the newest version (16+257build1.1).
postgresql-contrib is already the newest version (16+257build1.1).
0 upgraded, 0 newly installed, 0 to remove and 24 not upgraded.
 * Starting PostgreSQL 16 database server
   ...done.
                                                                 version                                                                  
------------------------------------------------------------------------------------------------------------------------------------------
 PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit
(1 row)



W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
%%bash
sudo -u postgres psql -U postgres -c "DROP DATABASE IF EXISTS casecompass_db;"
sudo -u postgres psql -U postgres -c "CREATE DATABASE casecompass_db;"

DROP DATABASE
CREATE DATABASE


In [ ]:
%%bash
sudo -u postgres psql -U postgres -c "DROP USER IF EXISTS casecompass_user;"
sudo -u postgres psql -U postgres -c "CREATE USER casecompass_user WITH PASSWORD 'casecompass_pass';"
sudo -u postgres psql -U postgres -c "GRANT ALL PRIVILEGES ON DATABASE casecompass_db TO casecompass_user;"

DROP ROLE
CREATE ROLE
GRANT


NOTICE:  role "casecompass_user" does not exist, skipping


In [ ]:
%%bash
sudo -u postgres psql -U postgres -d casecompass_db -c "
GRANT ALL ON SCHEMA public TO casecompass_user;
ALTER DEFAULT PRIVILEGES IN SCHEMA public
GRANT ALL ON TABLES TO casecompass_user;
ALTER DEFAULT PRIVILEGES IN SCHEMA public
GRANT ALL ON SEQUENCES TO casecompass_user;
"

GRANT
ALTER DEFAULT PRIVILEGES
ALTER DEFAULT PRIVILEGES


In [ ]:
%%bash
sudo -u postgres psql -U postgres -c "\l casecompass_db"
sudo -u postgres psql -U postgres -c "\du casecompass_user"

                                                              List of databases
      Name      |  Owner   | Encoding | Locale Provider |   Collate   |    Ctype    | ICU Locale | ICU Rules |       Access privileges       
----------------+----------+----------+-----------------+-------------+-------------+------------+-----------+-------------------------------
 casecompass_db | postgres | UTF8     | libc            | en_US.UTF-8 | en_US.UTF-8 |            |           | =Tc/postgres                 +
                |          |          |                 |             |             |            |           | postgres=CTc/postgres        +
                |          |          |                 |             |             |            |           | casecompass_user=CTc/postgres
(1 row)

         List of roles
    Role name     | Attributes 
------------------+------------
 casecompass_user | 



In [ ]:
!pip -q install psycopg2-binary


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 80.1 MB/s eta 0:00:00


In [ ]:
import psycopg2

conn = psycopg2.connect(
    dbname="casecompass_db",
    user="casecompass_user",
    password="casecompass_pass",
    host="localhost",
    port="5432"
)

cur = conn.cursor()
cur.execute("SELECT current_database(), current_user;")
print("Connected:", cur.fetchone())

conn.close()


Connected: ('casecompass_db', 'casecompass_user')


In [ ]:
schema_sql = r"""
-- Paste the entire CaseCompass PostgreSQL schema here
"""

with open("casecompass_schema.sql", "w") as f:
    f.write(schema_sql)

print("Saved casecompass_schema.sql")

Saved casecompass_schema.sql


In [ ]:
import psycopg2

conn = psycopg2.connect(
    dbname="casecompass_db",
    user="casecompass_user",
    password="casecompass_pass",
    host="localhost",
    port="5432"
)
conn.autocommit = True
cur = conn.cursor()

with open("casecompass_schema.sql", "r") as f:
    schema_sql = f.read()

cur.execute(schema_sql)
print("CaseCompass schema created successfully")

conn.close()

ProgrammingError: can't execute an empty query

In [ ]:
import psycopg2

conn = psycopg2.connect(
    dbname="casecompass_db",
    user="casecompass_user",
    password="casecompass_pass",
    host="localhost",
    port="5432"
)
conn.autocommit = True
cur = conn.cursor()

with open("casecompass_schema.sql", "r") as f:
    sql_text = f.read()

statements = [s.strip() for s in sql_text.split(";") if s.strip()]

for i, statement in enumerate(statements, 1):
    try:
        cur.execute(statement)
        print(f"OK {i}: {' '.join(statement.split())[:80]}...")
    except Exception as e:
        print("\nFAILED AT STATEMENT", i)
        print(statement)
        print("\nERROR:")
        print(e)
        break

conn.close()


FAILED AT STATEMENT 1
-- Paste the entire CaseCompass PostgreSQL schema here

ERROR:
can't execute an empty query


In [ ]:
import psycopg2

conn = psycopg2.connect(
    dbname="casecompass_db",
    user="casecompass_user",
    password="casecompass_pass",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

cur.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
""")

tables = [row[0] for row in cur.fetchall()]
print("Tables created:")
for table in tables:
    print("-", table)

conn.close()

Tables created:


In [ ]:
import psycopg2
from datetime import date

conn = psycopg2.connect(
    dbname="casecompass_db",
    user="casecompass_user",
    password="casecompass_pass",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

# Create case
cur.execute("""
    INSERT INTO cases (case_reference, title, jurisdiction, court_or_tribunal)
    VALUES (%s, %s, %s, %s)
    RETURNING case_id;
""", (
    "CASE-DEMO-001",
    "Fictional Rental Dispute",
    "Fictional jurisdiction",
    "Fictional District Court"
))
case_id = cur.fetchone()[0]

# Create document
cur.execute("""
    INSERT INTO documents (
        case_id, document_type, title, file_name,
        document_date_raw, document_date, date_confidence, status
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    RETURNING document_id;
""", (
    case_id,
    "court_order",
    "Interim Order dated 12 March 2026",
    "interim_order.pdf",
    "12 March 2026",
    date(2026, 3, 12),
    "high",
    "processed"
))
document_id = cur.fetchone()[0]

# Create page
cur.execute("""
    INSERT INTO document_pages (document_id, page_number, page_text)
    VALUES (%s, %s, %s)
    RETURNING page_id;
""", (
    document_id,
    1,
    "The respondent shall file a written reply by 2 April 2026. "
    "This order refers to Order No. 14 dated 5 February 2026."
))
page_id = cur.fetchone()[0]

# Create exact source span
page_text = (
    "The respondent shall file a written reply by 2 April 2026. "
    "This order refers to Order No. 14 dated 5 February 2026."
)

cur.execute("""
    INSERT INTO source_spans (
        document_id, page_id, start_offset, end_offset,
        source_text, extraction_method
    )
    VALUES (%s, %s, %s, %s, %s, %s)
    RETURNING source_span_id;
""", (
    document_id,
    page_id,
    0,
    len(page_text),
    page_text,
    "manual"
))
source_span_id = cur.fetchone()[0]

# Create reviewed event
cur.execute("""
    INSERT INTO case_events (
        case_id, event_type, title, description,
        event_date, event_date_raw, date_confidence,
        sequence_hint, is_order, review_status, created_by
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    RETURNING event_id;
""", (
    case_id,
    "court_order",
    "Interim order issued",
    "The court issued an interim order requiring a written reply.",
    date(2026, 3, 12),
    "12 March 2026",
    "high",
    1,
    True,
    "confirmed",
    "demo_user"
))
event_id = cur.fetchone()[0]

# Link event to evidence
cur.execute("""
    INSERT INTO event_assertions (
        case_id, event_id, document_id, source_span_id,
        asserted_event, asserted_date_raw, asserted_date,
        date_confidence, assertion_role, ai_generated, review_status
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
""", (
    case_id,
    event_id,
    document_id,
    source_span_id,
    "Interim order issued",
    "12 March 2026",
    date(2026, 3, 12),
    "high",
    "primary",
    False,
    "confirmed"
))

# Create explicit deadline
cur.execute("""
    INSERT INTO case_deadlines (
        case_id, event_id, title, action_required,
        due_date_raw, due_date, date_confidence,
        deadline_basis, review_status
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    RETURNING deadline_id;
""", (
    case_id,
    event_id,
    "File written reply",
    "File a written reply",
    "2 April 2026",
    date(2026, 4, 2),
    "high",
    "explicit_in_record",
    "confirmed"
))
deadline_id = cur.fetchone()[0]

# Link deadline to exact source sentence
cur.execute("""
    INSERT INTO deadline_assertions (
        deadline_id, document_id, source_span_id,
        stated_due_date_raw, stated_due_date,
        date_confidence, ai_generated, review_status
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s);
""", (
    deadline_id,
    document_id,
    source_span_id,
    "2 April 2026",
    date(2026, 4, 2),
    "high",
    False,
    "confirmed"
))

conn.commit()

print("Demo case inserted successfully")
print("case_id:", case_id)
print("document_id:", document_id)
print("event_id:", event_id)
print("deadline_id:", deadline_id)

conn.close()

UndefinedTable: relation "cases" does not exist
LINE 2:     INSERT INTO cases (case_reference, title, jurisdiction, ...
                        ^


In [ ]:
import psycopg2

conn = psycopg2.connect(
    dbname="casecompass_db",
    user="casecompass_user",
    password="casecompass_pass",
    host="localhost",
    port="5432"
)
conn.autocommit = True
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS cases (
    case_id UUID PRIMARY KEY
        DEFAULT md5(random()::text || clock_timestamp()::text)::uuid,
    case_reference TEXT UNIQUE NOT NULL,
    title TEXT NOT NULL,
    jurisdiction TEXT,
    court_or_tribunal TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    updated_at TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS documents (
    document_id UUID PRIMARY KEY
        DEFAULT md5(random()::text || clock_timestamp()::text)::uuid,
    case_id UUID NOT NULL REFERENCES cases(case_id) ON DELETE CASCADE,
    document_type TEXT NOT NULL,
    title TEXT NOT NULL,
    file_name TEXT,
    storage_uri TEXT,
    sha256_hash TEXT,
    document_date_raw TEXT,
    document_date DATE,
    date_confidence TEXT NOT NULL DEFAULT 'unparsed',
    status TEXT NOT NULL DEFAULT 'uploaded',
    uploaded_at TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS document_pages (
    page_id UUID PRIMARY KEY
        DEFAULT md5(random()::text || clock_timestamp()::text)::uuid,
    document_id UUID NOT NULL REFERENCES documents(document_id) ON DELETE CASCADE,
    page_number INTEGER NOT NULL,
    page_text TEXT NOT NULL,
    ocr_confidence NUMERIC(4,3),
    created_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    UNIQUE (document_id, page_number)
);

CREATE TABLE IF NOT EXISTS source_spans (
    source_span_id UUID PRIMARY KEY
        DEFAULT md5(random()::text || clock_timestamp()::text)::uuid,
    document_id UUID NOT NULL REFERENCES documents(document_id) ON DELETE CASCADE,
    page_id UUID REFERENCES document_pages(page_id) ON DELETE SET NULL,
    start_offset INTEGER NOT NULL,
    end_offset INTEGER NOT NULL,
    source_text TEXT NOT NULL,
    extraction_method TEXT NOT NULL DEFAULT 'manual',
    created_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    CHECK (start_offset >= 0 AND end_offset > start_offset)
);

CREATE TABLE IF NOT EXISTS case_events (
    event_id UUID PRIMARY KEY
        DEFAULT md5(random()::text || clock_timestamp()::text)::uuid,
    case_id UUID NOT NULL REFERENCES cases(case_id) ON DELETE CASCADE,
    event_type TEXT NOT NULL,
    title TEXT NOT NULL,
    description TEXT,
    event_date DATE,
    event_date_raw TEXT,
    date_confidence TEXT NOT NULL DEFAULT 'unparsed',
    sequence_hint INTEGER,
    is_order BOOLEAN NOT NULL DEFAULT false,
    order_number TEXT,
    review_status TEXT NOT NULL DEFAULT 'pending',
    created_by TEXT NOT NULL DEFAULT 'system',
    created_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    updated_at TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS event_assertions (
    assertion_id UUID PRIMARY KEY
        DEFAULT md5(random()::text || clock_timestamp()::text)::uuid,
    case_id UUID NOT NULL REFERENCES cases(case_id) ON DELETE CASCADE,
    event_id UUID REFERENCES case_events(event_id) ON DELETE SET NULL,
    document_id UUID NOT NULL REFERENCES documents(document_id) ON DELETE CASCADE,
    source_span_id UUID NOT NULL REFERENCES source_spans(source_span_id) ON DELETE RESTRICT,
    asserted_event TEXT NOT NULL,
    asserted_date_raw TEXT,
    asserted_date DATE,
    date_confidence TEXT NOT NULL DEFAULT 'unparsed',
    assertion_role TEXT NOT NULL DEFAULT 'primary',
    ai_generated BOOLEAN NOT NULL DEFAULT false,
    review_status TEXT NOT NULL DEFAULT 'pending',
    created_at TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS case_deadlines (
    deadline_id UUID PRIMARY KEY
        DEFAULT md5(random()::text || clock_timestamp()::text)::uuid,
    case_id UUID NOT NULL REFERENCES cases(case_id) ON DELETE CASCADE,
    event_id UUID REFERENCES case_events(event_id) ON DELETE SET NULL,
    title TEXT NOT NULL,
    action_required TEXT NOT NULL,
    due_date_raw TEXT NOT NULL,
    due_date DATE,
    date_confidence TEXT NOT NULL DEFAULT 'unparsed',
    deadline_basis TEXT NOT NULL,
    user_rule_id UUID,
    review_status TEXT NOT NULL DEFAULT 'pending',
    created_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    updated_at TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS deadline_assertions (
    deadline_assertion_id UUID PRIMARY KEY
        DEFAULT md5(random()::text || clock_timestamp()::text)::uuid,
    deadline_id UUID NOT NULL REFERENCES case_deadlines(deadline_id) ON DELETE CASCADE,
    document_id UUID NOT NULL REFERENCES documents(document_id) ON DELETE CASCADE,
    source_span_id UUID NOT NULL REFERENCES source_spans(source_span_id) ON DELETE RESTRICT,
    stated_due_date_raw TEXT NOT NULL,
    stated_due_date DATE,
    date_confidence TEXT NOT NULL DEFAULT 'unparsed',
    ai_generated BOOLEAN NOT NULL DEFAULT false,
    review_status TEXT NOT NULL DEFAULT 'pending',
    created_at TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS timeline_gaps (
    gap_id UUID PRIMARY KEY
        DEFAULT md5(random()::text || clock_timestamp()::text)::uuid,
    case_id UUID NOT NULL REFERENCES cases(case_id) ON DELETE CASCADE,
    gap_type TEXT NOT NULL,
    related_event_id UUID REFERENCES case_events(event_id) ON DELETE SET NULL,
    related_document_id UUID REFERENCES documents(document_id) ON DELETE SET NULL,
    related_order_reference_id UUID,
    related_conflict_id UUID,
    gap_description TEXT NOT NULL,
    severity TEXT NOT NULL DEFAULT 'medium',
    status TEXT NOT NULL DEFAULT 'open',
    created_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    resolved_at TIMESTAMPTZ,
    resolution_note TEXT
);
""")

print("CaseCompass tables created successfully")
conn.close()

CaseCompass tables created successfully


In [ ]:
import psycopg2

conn = psycopg2.connect(
    dbname="casecompass_db",
    user="casecompass_user",
    password="casecompass_pass",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

cur.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
      AND table_name IN (
          'cases', 'documents', 'document_pages', 'source_spans',
          'case_events', 'event_assertions', 'case_deadlines',
          'deadline_assertions', 'timeline_gaps'
      )
    ORDER BY table_name;
""")

for (table,) in cur.fetchall():
    print("-", table)

conn.close()

- case_deadlines
- case_events
- cases
- deadline_assertions
- document_pages
- documents
- event_assertions
- source_spans
- timeline_gaps


In [ ]:
import psycopg2

conn = psycopg2.connect(
    dbname="casecompass_db",
    user="casecompass_user",
    password="casecompass_pass",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

cur.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
""")

for (table_name,) in cur.fetchall():
    print(table_name)

conn.close()

case_deadlines
case_events
cases
deadline_assertions
document_pages
documents
event_assertions
source_spans
timeline_gaps


In [ ]:
import psycopg2
from datetime import date

conn = psycopg2.connect(
    dbname="casecompass_db",
    user="casecompass_user",
    password="casecompass_pass",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

# Create case
cur.execute("""
    INSERT INTO cases (case_reference, title, jurisdiction, court_or_tribunal)
    VALUES (%s, %s, %s, %s)
    RETURNING case_id;
""", (
    "CASE-DEMO-001",
    "Fictional Rental Dispute",
    "Fictional jurisdiction",
    "Fictional District Court"
))
case_id = cur.fetchone()[0]

In [ ]:
import psycopg2

conn = psycopg2.connect(
    dbname="casecompass_db",
    user="casecompass_user",
    password="casecompass_pass",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

cur.execute("""
    SELECT case_reference, title
    FROM cases
    WHERE case_reference = 'CASE-DEMO-001';
""")

result = cur.fetchone()

if result:
    print("Demo case already exists:")
    print("Reference:", result[0])
    print("Title:", result[1])
else:
    print("No demo case found. Run the complete demo insertion cell.")

conn.close()

No demo case found. Run the complete demo insertion cell.


In [ ]:
import psycopg2
from datetime import date

conn = psycopg2.connect(
    dbname="casecompass_db",
    user="casecompass_user",
    password="casecompass_pass",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

# 1. Create case
cur.execute("""
    INSERT INTO cases (case_reference, title, jurisdiction, court_or_tribunal)
    VALUES (%s, %s, %s, %s)
    RETURNING case_id;
""", (
    "CASE-DEMO-001",
    "Fictional Rental Dispute",
    "Fictional jurisdiction",
    "Fictional District Court"
))
case_id = cur.fetchone()[0]
print("Case created:", case_id)

# 2. Create document
cur.execute("""
    INSERT INTO documents (
        case_id, document_type, title, file_name,
        document_date_raw, document_date, date_confidence, status
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    RETURNING document_id;
""", (
    case_id,
    "court_order",
    "Interim Order dated 12 March 2026",
    "interim_order.pdf",
    "12 March 2026",
    date(2026, 3, 12),
    "high",
    "processed"
))
document_id = cur.fetchone()[0]
print("Document created:", document_id)

# 3. Create document page
page_text = (
    "The respondent shall file a written reply by 2 April 2026. "
    "This order refers to Order No. 14 dated 5 February 2026."
)

cur.execute("""
    INSERT INTO document_pages (document_id, page_number, page_text)
    VALUES (%s, %s, %s)
    RETURNING page_id;
""", (
    document_id,
    1,
    page_text
))
page_id = cur.fetchone()[0]
print("Page created:", page_id)

# 4. Create exact source span
cur.execute("""
    INSERT INTO source_spans (
        document_id, page_id, start_offset, end_offset,
        source_text, extraction_method
    )
    VALUES (%s, %s, %s, %s, %s, %s)
    RETURNING source_span_id;
""", (
    document_id,
    page_id,
    0,
    len(page_text),
    page_text,
    "manual"
))
source_span_id = cur.fetchone()[0]
print("Source span created:", source_span_id)

# 5. Create timeline event
cur.execute("""
    INSERT INTO case_events (
        case_id, event_type, title, description,
        event_date, event_date_raw, date_confidence,
        sequence_hint, is_order, review_status, created_by
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    RETURNING event_id;
""", (
    case_id,
    "court_order",
    "Interim order issued",
    "The court issued an interim order requiring a written reply.",
    date(2026, 3, 12),
    "12 March 2026",
    "high",
    1,
    True,
    "confirmed",
    "demo_user"
))
event_id = cur.fetchone()[0]
print("Event created:", event_id)

# 6. Link event to evidence
cur.execute("""
    INSERT INTO event_assertions (
        case_id, event_id, document_id, source_span_id,
        asserted_event, asserted_date_raw, asserted_date,
        date_confidence, assertion_role, ai_generated, review_status
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
""", (
    case_id,
    event_id,
    document_id,
    source_span_id,
    "Interim order issued",
    "12 March 2026",
    date(2026, 3, 12),
    "high",
    "primary",
    False,
    "confirmed"
))
print("Event evidence linked")

# 7. Create explicit deadline
cur.execute("""
    INSERT INTO case_deadlines (
        case_id, event_id, title, action_required,
        due_date_raw, due_date, date_confidence,
        deadline_basis, review_status
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    RETURNING deadline_id;
""", (
    case_id,
    event_id,
    "File written reply",
    "File a written reply",
    "2 April 2026",
    date(2026, 4, 2),
    "high",
    "explicit_in_record",
    "confirmed"
))
deadline_id = cur.fetchone()[0]
print("Deadline created:", deadline_id)

# 8. Link deadline to exact source sentence
cur.execute("""
    INSERT INTO deadline_assertions (
        deadline_id, document_id, source_span_id,
        stated_due_date_raw, stated_due_date,
        date_confidence, ai_generated, review_status
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s);
""", (
    deadline_id,
    document_id,
    source_span_id,
    "2 April 2026",
    date(2026, 4, 2),
    "high",
    False,
    "confirmed"
))
print("Deadline evidence linked")

# 9. Save everything permanently
conn.commit()

print("\nDemo case inserted successfully")
print("case_id:", case_id)
print("document_id:", document_id)
print("event_id:", event_id)
print("deadline_id:", deadline_id)

cur.close()
conn.close()

Case created: 1297cc1a-dfa2-7f3f-e88c-6414e2f63712
Document created: 94ab2fe8-437c-c1b3-b59b-58ffb95cc198
Page created: 5aad8cac-bc08-b077-8731-a262672fe4ec
Source span created: 8f5fe848-1607-59d4-2316-5da91906ff80
Event created: 9075b65c-10b2-da56-b258-49f868567bcc
Event evidence linked
Deadline created: 7971822e-3820-6d4b-515e-cd219c142f29
Deadline evidence linked

Demo case inserted successfully
case_id: 1297cc1a-dfa2-7f3f-e88c-6414e2f63712
document_id: 94ab2fe8-437c-c1b3-b59b-58ffb95cc198
event_id: 9075b65c-10b2-da56-b258-49f868567bcc
deadline_id: 7971822e-3820-6d4b-515e-cd219c142f29


In [ ]:
import psycopg2

conn = psycopg2.connect(
    dbname="casecompass_db",
    user="casecompass_user",
    password="casecompass_pass",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

cur.execute("""
    SELECT
        ce.title,
        COALESCE(ce.event_date::TEXT, ce.event_date_raw) AS date_shown,
        ce.date_confidence,
        d.title AS source_document,
        ss.source_text AS evidence
    FROM case_events ce
    JOIN event_assertions ea
        ON ea.event_id = ce.event_id
       AND ea.review_status = 'confirmed'
    JOIN source_spans ss
        ON ss.source_span_id = ea.source_span_id
    JOIN documents d
        ON d.document_id = ss.document_id
    WHERE ce.case_id = (
        SELECT case_id
        FROM cases
        WHERE case_reference = 'CASE-DEMO-001'
    )
    ORDER BY ce.event_date NULLS LAST;
""")

rows = cur.fetchall()

print("=== CASE TIMELINE ===")
for title, date_shown, confidence, document, evidence in rows:
    print("\nEVENT:", title)
    print("DATE:", date_shown, "| CONFIDENCE:", confidence)
    print("SOURCE DOCUMENT:", document)
    print("EVIDENCE:", evidence)

cur.close()
conn.close()

=== CASE TIMELINE ===

EVENT: Interim order issued
DATE: 2026-03-12 | CONFIDENCE: high
SOURCE DOCUMENT: Interim Order dated 12 March 2026
EVIDENCE: The respondent shall file a written reply by 2 April 2026. This order refers to Order No. 14 dated 5 February 2026.


In [ ]:
import psycopg2

conn = psycopg2.connect(
    dbname="casecompass_db",
    user="casecompass_user",
    password="casecompass_pass",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

cur.execute("""
    SELECT
        cd.title,
        cd.action_required,
        COALESCE(cd.due_date::TEXT, cd.due_date_raw) AS due_date_shown,
        cd.date_confidence,
        d.title AS source_document,
        ss.source_text AS exact_source_sentence
    FROM case_deadlines cd
    JOIN deadline_assertions da
        ON da.deadline_id = cd.deadline_id
       AND da.review_status = 'confirmed'
    JOIN source_spans ss
        ON ss.source_span_id = da.source_span_id
    JOIN documents d
        ON d.document_id = ss.document_id
    WHERE cd.case_id = (
        SELECT case_id
        FROM cases
        WHERE case_reference = 'CASE-DEMO-001'
    )
    ORDER BY cd.due_date NULLS LAST;
""")

rows = cur.fetchall()

print("=== EXPLICIT DEADLINES ===")

for title, action, due_date, confidence, document, evidence in rows:
    print("\nDEADLINE:", title)
    print("ACTION:", action)
    print("DUE DATE:", due_date, "| CONFIDENCE:", confidence)
    print("SOURCE DOCUMENT:", document)
    print("EXACT SOURCE:", evidence)

cur.close()
conn.close()

=== EXPLICIT DEADLINES ===

DEADLINE: File written reply
ACTION: File a written reply
DUE DATE: 2026-04-02 | CONFIDENCE: high
SOURCE DOCUMENT: Interim Order dated 12 March 2026
EXACT SOURCE: The respondent shall file a written reply by 2 April 2026. This order refers to Order No. 14 dated 5 February 2026.


In [ ]:
import psycopg2

conn = psycopg2.connect(
    dbname="casecompass_db",
    user="casecompass_user",
    password="casecompass_pass",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

cur.execute("""
    INSERT INTO timeline_gaps (
        case_id, gap_type, related_document_id,
        gap_description, severity
    )
    SELECT
        c.case_id,
        'missing_referenced_order',
        d.document_id,
        'The document refers to Order No. 14 dated 5 February 2026, but that order was not supplied.',
        'high'
    FROM cases c
    JOIN documents d ON d.case_id = c.case_id
    WHERE c.case_reference = 'CASE-DEMO-001'
      AND d.title = 'Interim Order dated 12 March 2026'
    RETURNING gap_id;
""")

gap_id = cur.fetchone()[0]
conn.commit()

print("Missing-order gap created:", gap_id)

cur.close()
conn.close()

Missing-order gap created: ab684674-3ce9-85d0-75df-52ed36a2db3f


In [ ]:
import psycopg2

conn = psycopg2.connect(
    dbname="casecompass_db",
    user="casecompass_user",
    password="casecompass_pass",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

cur.execute("""
    SELECT
        tg.gap_type,
        tg.gap_description,
        tg.severity,
        tg.status,
        d.title AS source_document
    FROM timeline_gaps tg
    JOIN cases c ON c.case_id = tg.case_id
    LEFT JOIN documents d ON d.document_id = tg.related_document_id
    WHERE c.case_reference = 'CASE-DEMO-001'
    ORDER BY tg.created_at DESC;
""")

rows = cur.fetchall()

print("=== TIMELINE GAPS ===")

for gap_type, description, severity, status, document in rows:
    print("\nGAP TYPE:", gap_type)
    print("DESCRIPTION:", description)
    print("SEVERITY:", severity)
    print("STATUS:", status)
    print("SOURCE DOCUMENT:", document)

cur.close()
conn.close()

=== TIMELINE GAPS ===

GAP TYPE: missing_referenced_order
DESCRIPTION: The document refers to Order No. 14 dated 5 February 2026, but that order was not supplied.
SEVERITY: high
STATUS: open
SOURCE DOCUMENT: Interim Order dated 12 March 2026


In [ ]:
!pip -q install fastapi uvicorn psycopg2-binary


In [ ]:
%%writefile main.py
from fastapi import FastAPI, HTTPException
import psycopg2
import os

app = FastAPI(
    title="CaseCompass API",
    description=(
        "Organizes supplied legal-case records into a source-linked timeline. "
        "Does not provide legal advice or predict outcomes."
    )
)

DB = {
    "dbname": "casecompass_db",
    "user": "casecompass_user",
    "password": "casecompass_pass",
    "host": "localhost",
    "port": "5432"
}

def get_cursor():
    conn = psycopg2.connect(**DB)
    return conn, conn.cursor()

@app.get("/")
def home():
    return {
        "product": "CaseCompass",
        "message": (
            "CaseCompass organizes supplied case records. "
            "It does not provide legal advice, predict outcomes, "
            "or calculate deadlines unless the user supplies a rule."
        )
    }

@app.get("/cases/{case_reference}/timeline")
def get_timeline(case_reference: str):
    conn, cur = get_cursor()

    cur.execute("""
        SELECT
            ce.title,
            ce.event_type,
            COALESCE(ce.event_date::TEXT, ce.event_date_raw) AS date_shown,
            ce.date_confidence,
            d.title AS source_document,
            ss.source_text AS evidence
        FROM case_events ce
        JOIN event_assertions ea
            ON ea.event_id = ce.event_id
           AND ea.review_status = 'confirmed'
        JOIN source_spans ss
            ON ss.source_span_id = ea.source_span_id
        JOIN documents d
            ON d.document_id = ss.document_id
        WHERE ce.case_id = (
            SELECT case_id FROM cases WHERE case_reference = %s
        )
        ORDER BY ce.event_date NULLS LAST;
    """, (case_reference,))

    rows = cur.fetchall()
    cur.close()
    conn.close()

    if not rows:
        raise HTTPException(status_code=404, detail="No timeline events found")

    return {
        "case_reference": case_reference,
        "timeline": [
            {
                "event": row[0],
                "event_type": row[1],
                "date": row[2],
                "date_confidence": row[3],
                "source_document": row[4],
                "evidence": row[5]
            }
            for row in rows
        ]
    }

@app.get("/cases/{case_reference}/deadlines")
def get_deadlines(case_reference: str):
    conn, cur = get_cursor()

    cur.execute("""
        SELECT
            cd.title,
            cd.action_required,
            COALESCE(cd.due_date::TEXT, cd.due_date_raw) AS due_date_shown,
            cd.date_confidence,
            d.title AS source_document,
            ss.source_text AS exact_source_sentence
        FROM case_deadlines cd
        JOIN deadline_assertions da
            ON da.deadline_id = cd.deadline_id
           AND da.review_status = 'confirmed'
        JOIN source_spans ss
            ON ss.source_span_id = da.source_span_id
        JOIN documents d
            ON d.document_id = ss.document_id
        WHERE cd.case_id = (
            SELECT case_id FROM cases WHERE case_reference = %s
        )
        ORDER BY cd.due_date NULLS LAST;
    """, (case_reference,))

    rows = cur.fetchall()
    cur.close()
    conn.close()

    if not rows:
        raise HTTPException(status_code=404, detail="No explicit deadlines found")

    return {
        "case_reference": case_reference,
        "deadlines": [
            {
                "deadline": row[0],
                "action_required": row[1],
                "due_date": row[2],
                "date_confidence": row[3],
                "source_document": row[4],
                "exact_source_sentence": row[5]
            }
            for row in rows
        ]
    }

@app.get("/cases/{case_reference}/gaps")
def get_gaps(case_reference: str):
    conn, cur = get_cursor()

    cur.execute("""
        SELECT
            tg.gap_type,
            tg.gap_description,
            tg.severity,
            tg.status,
            d.title AS source_document
        FROM timeline_gaps tg
        JOIN cases c ON c.case_id = tg.case_id
        LEFT JOIN documents d
            ON d.document_id = tg.related_document_id
        WHERE c.case_reference = %s
        ORDER BY tg.created_at DESC;
    """, (case_reference,))

    rows = cur.fetchall()
    cur.close()
    conn.close()

    return {
        "case_reference": case_reference,
        "gaps": [
            {
                "gap_type": row[0],
                "description": row[1],
                "severity": row[2],
                "status": row[3],
                "source_document": row[4]
            }
            for row in rows
        ]
    }

Writing main.py


In [ ]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_api():
    uvicorn.run("main:app", host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_api, daemon=True)
thread.start()

print("CaseCompass API running at http://127.0.0.1:8000")


CaseCompass API running at http://127.0.0.1:8000


In [ ]:
import requests

base = "http://127.0.0.1:8000"

print(requests.get(f"{base}/").json())
print("\nTIMELINE:")
print(requests.get(f"{base}/cases/CASE-DEMO-001/timeline").json())

print("\nDEADLINES:")
print(requests.get(f"{base}/cases/CASE-DEMO-001/deadlines").json())

print("\nGAPS:")
print(requests.get(f"{base}/cases/CASE-DEMO-001/gaps").json())

INFO:     127.0.0.1:35668 - "GET / HTTP/1.1" 200 OK
{'product': 'CaseCompass', 'message': 'CaseCompass organizes supplied case records. It does not provide legal advice, predict outcomes, or calculate deadlines unless the user supplies a rule.'}

TIMELINE:
INFO:     127.0.0.1:35670 - "GET /cases/CASE-DEMO-001/timeline HTTP/1.1" 200 OK
{'case_reference': 'CASE-DEMO-001', 'timeline': [{'event': 'Interim order issued', 'event_type': 'court_order', 'date': '2026-03-12', 'date_confidence': 'high', 'source_document': 'Interim Order dated 12 March 2026', 'evidence': 'The respondent shall file a written reply by 2 April 2026. This order refers to Order No. 14 dated 5 February 2026.'}]}

DEADLINES:
INFO:     127.0.0.1:35672 - "GET /cases/CASE-DEMO-001/deadlines HTTP/1.1" 200 OK
{'case_reference': 'CASE-DEMO-001', 'deadlines': [{'deadline': 'File written reply', 'action_required': 'File a written reply', 'due_date': '2026-04-02', 'date_confidence': 'high', 'source_document': 'Interim Order dated

In [ ]:
!pip -q install fastapi uvicorn psycopg2-binary

In [ ]:
%%writefile main.py
from fastapi import FastAPI, HTTPException
import psycopg2
import os

app = FastAPI(
    title="CaseCompass API",
    description=(
        "Organizes supplied legal-case records into a source-linked timeline. "
        "Does not provide legal advice or predict outcomes."
    )
)

DB = {
    "dbname": "casecompass_db",
    "user": "casecompass_user",
    "password": "casecompass_pass",
    "host": "localhost",
    "port": "5432"
}

def get_cursor():
    conn = psycopg2.connect(**DB)
    return conn, conn.cursor()

@app.get("/")
def home():
    return {
        "product": "CaseCompass",
        "message": (
            "CaseCompass organizes supplied case records. "
            "It does not provide legal advice, predict outcomes, "
            "or calculate deadlines unless the user supplies a rule."
        )
    }

@app.get("/cases/{case_reference}/timeline")
def get_timeline(case_reference: str):
    conn, cur = get_cursor()

    cur.execute("""
        SELECT
            ce.title,
            ce.event_type,
            COALESCE(ce.event_date::TEXT, ce.event_date_raw) AS date_shown,
            ce.date_confidence,
            d.title AS source_document,
            ss.source_text AS evidence
        FROM case_events ce
        JOIN event_assertions ea
            ON ea.event_id = ce.event_id
           AND ea.review_status = 'confirmed'
        JOIN source_spans ss
            ON ss.source_span_id = ea.source_span_id
        JOIN documents d
            ON d.document_id = ss.document_id
        WHERE ce.case_id = (
            SELECT case_id FROM cases WHERE case_reference = %s
        )
        ORDER BY ce.event_date NULLS LAST;
    """, (case_reference,))

    rows = cur.fetchall()
    cur.close()
    conn.close()

    if not rows:
        raise HTTPException(status_code=404, detail="No timeline events found")

    return {
        "case_reference": case_reference,
        "timeline": [
            {
                "event": row[0],
                "event_type": row[1],
                "date": row[2],
                "date_confidence": row[3],
                "source_document": row[4],
                "evidence": row[5]
            }
            for row in rows
        ]
    }

@app.get("/cases/{case_reference}/deadlines")
def get_deadlines(case_reference: str):
    conn, cur = get_cursor()

    cur.execute("""
        SELECT
            cd.title,
            cd.action_required,
            COALESCE(cd.due_date::TEXT, cd.due_date_raw) AS due_date_shown,
            cd.date_confidence,
            d.title AS source_document,
            ss.source_text AS exact_source_sentence
        FROM case_deadlines cd
        JOIN deadline_assertions da
            ON da.deadline_id = cd.deadline_id
           AND da.review_status = 'confirmed'
        JOIN source_spans ss
            ON ss.source_span_id = da.source_span_id
        JOIN documents d
            ON d.document_id = ss.document_id
        WHERE cd.case_id = (
            SELECT case_id FROM cases WHERE case_reference = %s
        )
        ORDER BY cd.due_date NULLS LAST;
    """, (case_reference,))

    rows = cur.fetchall()
    cur.close()
    conn.close()

    if not rows:
        raise HTTPException(status_code=404, detail="No explicit deadlines found")

    return {
        "case_reference": case_reference,
        "deadlines": [
            {
                "deadline": row[0],
                "action_required": row[1],
                "due_date": row[2],
                "date_confidence": row[3],
                "source_document": row[4],
                "exact_source_sentence": row[5]
            }
            for row in rows
        ]
    }

@app.get("/cases/{case_reference}/gaps")
def get_gaps(case_reference: str):
    conn, cur = get_cursor()

    cur.execute("""
        SELECT
            tg.gap_type,
            tg.gap_description,
            tg.severity,
            tg.status,
            d.title AS source_document
        FROM timeline_gaps tg
        JOIN cases c ON c.case_id = tg.case_id
        LEFT JOIN documents d
            ON d.document_id = tg.related_document_id
        WHERE c.case_reference = %s
        ORDER BY tg.created_at DESC;
    """, (case_reference,))

    rows = cur.fetchall()
    cur.close()
    conn.close()

    return {
        "case_reference": case_reference,
        "gaps": [
            {
                "gap_type": row[0],
                "description": row[1],
                "severity": row[2],
                "status": row[3],
                "source_document": row[4]
            }
            for row in rows
        ]
    }

Overwriting main.py


In [ ]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_api():
    uvicorn.run("main:app", host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_api, daemon=True)
thread.start()

print("CaseCompass API running at http://127.0.0.1:8000")

INFO:     Started server process [1071]


CaseCompass API running at http://127.0.0.1:8000


INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


In [ ]:
import requests

base = "http://127.0.0.1:8000"

print("HOME:")
print(requests.get(f"{base}/").json())

print("\nTIMELINE:")
print(requests.get(f"{base}/cases/CASE-DEMO-001/timeline").json())

print("\nDEADLINES:")
print(requests.get(f"{base}/cases/CASE-DEMO-001/deadlines").json())

print("\nGAPS:")
print(requests.get(f"{base}/cases/CASE-DEMO-001/gaps").json())

HOME:
INFO:     127.0.0.1:51530 - "GET / HTTP/1.1" 200 OK
{'product': 'CaseCompass', 'message': 'CaseCompass organizes supplied case records. It does not provide legal advice, predict outcomes, or calculate deadlines unless the user supplies a rule.'}

TIMELINE:
INFO:     127.0.0.1:51544 - "GET /cases/CASE-DEMO-001/timeline HTTP/1.1" 200 OK
{'case_reference': 'CASE-DEMO-001', 'timeline': [{'event': 'Interim order issued', 'event_type': 'court_order', 'date': '2026-03-12', 'date_confidence': 'high', 'source_document': 'Interim Order dated 12 March 2026', 'evidence': 'The respondent shall file a written reply by 2 April 2026. This order refers to Order No. 14 dated 5 February 2026.'}]}

DEADLINES:
INFO:     127.0.0.1:51560 - "GET /cases/CASE-DEMO-001/deadlines HTTP/1.1" 200 OK
{'case_reference': 'CASE-DEMO-001', 'deadlines': [{'deadline': 'File written reply', 'action_required': 'File a written reply', 'due_date': '2026-04-02', 'date_confidence': 'high', 'source_document': 'Interim Order

In [ ]:
%%writefile app.py
import streamlit as st
import requests

API = "http://127.0.0.1:8000"

st.set_page_config(page_title="CaseCompass", page_icon="⚖️")
st.title("⚖️ CaseCompass")
st.caption(
    "Organizes supplied case records into a source-linked timeline. "
    "Not legal advice. Does not predict outcomes or calculate deadlines."
)

case_reference = st.text_input("Case reference", value="CASE-DEMO-001")

tab1, tab2, tab3 = st.tabs(["📅 Timeline", "⏰ Deadlines", "⚠️ Gaps"])

with tab1:
    if st.button("Load timeline"):
        response = requests.get(f"{API}/cases/{case_reference}/timeline")

        if response.status_code == 200:
            for item in response.json()["timeline"]:
                st.subheader(item["event"])
                st.write(f"**Date:** {item['date']}")
                st.write(f"**Confidence:** {item['date_confidence']}")
                st.write(f"**Source document:** {item['source_document']}")
                st.info(item["evidence"])
        else:
            st.error("No timeline found for this case.")

with tab2:
    if st.button("Load deadlines"):
        response = requests.get(f"{API}/cases/{case_reference}/deadlines")

        if response.status_code == 200:
            for item in response.json()["deadlines"]:
                st.subheader(item["deadline"])
                st.write(f"**Action required:** {item['action_required']}")
                st.write(f"**Due date:** {item['due_date']}")
                st.write(f"**Confidence:** {item['date_confidence']}")
                st.write(f"**Source document:** {item['source_document']}")
                st.info(item["exact_source_sentence"])
        else:
            st.error("No explicit deadlines found for this case.")

with tab3:
    if st.button("Load gaps"):
        response = requests.get(f"{API}/cases/{case_reference}/gaps")
        gaps = response.json()["gaps"]

        if gaps:
            for gap in gaps:
                st.warning(
                    f"**{gap['gap_type'].replace('_', ' ').title()}**\n\n"
                    f"{gap['description']}\n\n"
                    f"Severity: {gap['severity']} | Status: {gap['status']}"
                )
        else:
            st.success("No open gaps detected.")

Writing app.py


In [ ]:
!pip -q install streamlit requests pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 61.8 MB/s eta 0:00:00


In [ ]:
from pyngrok import ngrok

# Replace this with your own ngrok authtoken
ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN")

public_url = ngrok.connect(8501, "http")
print("CaseCompass UI:", public_url)

ERROR:pyngrok.process.ngrok:t=2026-10-11T10:02:40+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified is an ngrok v1 authtoken, but you're using ngrok v2.\nYour authtoken: YOUR_NGROK_AUTHTOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_106\r\n"
ERROR:pyngrok.process.ngrok:t=2026-10-11T10:02:40+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified is an ngrok v1 authtoken, but you're using ngrok v2.\nYour authtoken: YOUR_NGROK_AUTHTOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_106\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: The authtoken you specified is an ngrok v1 authtoken, but you're using ngrok v2.\nYour authtoken: YOUR_NGROK_AUTHTOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_106\r\n.